# FlexCNN Medical Physics - Notebook Template

This notebook demonstrates how to use the modular parameter system for FlexCNN.

**Key Features:**
- Import default parameters from `user_params.py`
- Override specific parameters without modifying source files
- Rebuild configurations after parameter changes
- Reload package code without kernel restart using `reload_package()`

## Setup Environment

In [1]:
import os
import sys
import shutil
import psutil
import subprocess
import webbrowser


def _detect_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _detect_colab()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Local directory for module access
    local_project_dir = "/content/Colab/Working"
    os.makedirs(local_project_dir, exist_ok=True)

    # Drive mount location
    drive_project_dir = "/content/drive/MyDrive/Colab/Working"

    # Copy modules from Drive to local
    for filename in ("build_dicts.py", "script_setup.py", "user_params.py"):
        src = os.path.join(drive_project_dir, filename)
        dst = os.path.join(local_project_dir, filename)
        if os.path.isfile(src):
            shutil.copy2(src, dst)

    # Add local directory to sys.path
    if local_project_dir not in sys.path:
        sys.path.insert(0, local_project_dir)
else:
    project_dir = os.getcwd()
    if project_dir not in sys.path:
        sys.path.insert(0, project_dir)

from script_setup import (
    sense_colab, sense_device, install_packages,
    setup_colab_environment, setup_local_environment,
    reload_package, refresh_repo
)

IN_COLAB = sense_colab()
print(f"Running in {'Colab' if IN_COLAB else 'Local'} environment")


# Setup environment (run once per session)
install_packages(IN_COLAB, ray_version=None)

if IN_COLAB:
    setup_colab_environment(
        github_username='petercl8',
        repo_name='FlexCNN_for_Medical_Physics',
        skip_git_update=False,
        force_fresh_clone=False  # Set to True if you get import errors after updating GitHub
    )
else:
    # Local setup - use 'walk' mode for fast reload
    setup_local_environment(
        repo_name='FlexCNN_for_Medical_Physics',
        mode='walk',
    )
# Test resources (function now available in global namespace)
list_compute_resources()

ModuleNotFoundError: No module named 'script_setup'

## Load Default Parameters

In [ ]:
# Import after environment setup (build_dicts depends on package being importable)
from build_dicts import build_all_dicts
from USER_PARAMS import get_params

# Get default parameters
params = get_params()

# Display some key parameters
print(f"Run mode: {params['run_mode']}")
print(f"Network type: {params['network_type']}")
print(f"Train SI: {params['train_SI']}")
print(f"Gen sino size: {params['gen_sino_size']}")
print(f"Gen image size: {params['gen_image_size']}")

## Setup Paths and Device

In [ ]:
# Set main project directory (function available from environment setup)
project_dirPath = params['project_dirPath'] = setup_project_dirs(
    IN_COLAB,
    params['project_local_dirPath'],
    params['project_colab_dirPath'],
    mount_colab_drive=True
)

# Set device
device = params['device_opt'] = sense_device(device=params['device_opt'])

print(f"Project directory: {project_dirPath}")
print(f"Device: {device}")

## Build Configuration Dictionaries

In [ ]:
# Build all dictionaries
all_dicts = build_all_dicts(params)

# Extract main dictionaries
config = all_dicts['config']
paths = all_dicts['paths']
settings = all_dicts['settings']
base_dirs = all_dicts['base_dirs']
tune_opts = all_dicts['tune_opts']
test_opts = all_dicts['test_opts']

print("✅ Configuration dictionaries built successfully!")
print(f"Config keys: {list(config.keys())}")

# Tensorboard

In [ ]:
print(paths['tune_storage_dirPath'])

In [ ]:
#search_folder = 'search-ACT-288-bilinear-288x257-padZeros-tunedSSIM' 
#search_folder = 'search-ACT-288-bilinear-288x218-padSino-tunedSSIM' 
#search_folder = 'search-ACT-288-bilinear-288x257-padSino-tunedSSIM-dropEnforced' 
#search_folder = 'search-ACT-288-bilinear-288x257-padSino-tunedSSIM' 
#search_folder = 'search-ACT-320-bilinear-288x257-padSino-tunedSSIM' 
#search_folder = 'search-ACT-320-bilinear-288x257-padSino-fill_1_skipNone-tunedSSIM' 
search_folder = 'search-RECON_SINO_IS-320-FORE_recon-bilinear-288x257-padSino-tunedSSIM'

log_folder = os.path.join(paths['tune_storage_dirPath'], search_folder)


# 🔹 Kill any existing TensorBoard processes
for proc in psutil.process_iter(['pid', 'name']):
    if 'tensorboard' in proc.info['name'].lower():
        print(f"Killing existing TensorBoard PID {proc.info['pid']}")
        proc.kill()

# 🔹 Start TensorBoard as a subprocess
port = 6008
tb_process = subprocess.Popen(
    ["tensorboard", f"--logdir={log_folder}", f"--port={port}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    shell=True
)

print(f"TensorBoard started on http://localhost:{port}")

# 🔹 Open in browser automatically
webbrowser.open(f"http://localhost:{port}")

# Scales

In [ ]:
#avg_activity = compute_average_activity_per_image(paths, dataset='train')
#scales1 = compute_quantitative_reconstruction_scale(paths, dataset='train', compute_sinogram_scale=False)
#scales2 = analyze_reconstruction_scale_distribution(paths, dataset='train', sample_mode='full', sample_size=1000, ratio_cap_multiple=None)
#sinogram_scale = compute_sinogram_to_image_scale(paths, dataset='train', sample_number=1000)

# Sinogram Alignment

## Pre-generated

In [ ]:
indexes = [13, 14, 15, 16, 17, 18, 19]

act_sino = torch.from_numpy(np.load(os.path.join(paths['data_dirPath'], 'train-highCountSino-bilinear-288x180.npy'), mmap_mode='r'))[indexes,1,:,:].unsqueeze(dim=1)*settings['act_sino_scale']
#atten_sino = torch.from_numpy(np.load(os.path.join(paths['data_dirPath'], 'train-attenSino-180x180.npy'), mmap_mode='r'))[indexes,0,:,:].unsqueeze(dim=1)*settings['atten_sino_scale']

show_single_unmatched_tensor(act_sino)
#show_multiple_matched_tensors(act_sino, atten_sino)
#show_single_unmatched_tensor(act_sino+atten_sino)
print(settings)

## On the fly

In [ ]:
# --- Refresh Repository ---
#refresh_repo()
visualize=False

if visualize==True:
    visualize_sinogram_alignment(
        paths,
        settings,
        num_examples=5,
        scale_num_examples=None,  # Set to a large number to get an accurate scale
        start_index=600,
        randomize=True,
        random_seed=1,
        fig_size=3,
        cmap='inferno',
        circle=False,
        theta_type='symmetrical',  # 'speed' matches pooled angular sampling
        # Unified resize/pad options (applied to both activity and attenuation)
        sino_resize_type='pool', # bilinear or pool
        sino_pad_type='zeros',
        sino_init_vert_cut=None, # 300 for most resized
        vert_pool_size=1,
        horiz_pool_size=1,
        bilinear_intermediate_size=(288, 257),
        sino_size=288,
        # Attenuation generation options
        atten_creation_pool_size=1,
    )

# Dataframes

## Plot Learning Curves

In [ ]:
#csv_file_name = 'frame-ACT-288-pool-288x257-padZeros-tunedSSIM-1p0Lr-600epochs.csv'
#csv_file_name = 'frame-ACT-288-bilinear-288x257-padZeros-tunedSSIM-1p0lr-600epochs.csv'
#csv_file_name = 'frame-ACT-288-pool-288x257-padZeros-tunedSSIM-0p3lr-800epochs.csv'
#csv_file_name = 'frame-ACT-288-bilinear-288x257-padZeros-tunedSSIM-0p3lr-800epochs.csv'
#csv_file_name = 'frame-ACT-288-bilinear-288x257-padSino-tunedSSIM-0p3lr-370epochs.csv'
#csv_file_name = 'frame-ACT-288-bilinear-288x218-padSino-tunedSSIM-0p3lr-800epochs.csv'
#csv_file_name = 'frame-ACT-288-bilinear-288x180-padSino-tunedSSIM-0p3lr-800epochs.csv'
#csv_file_name = 'frame-ACT-288-bilinear-288x257-padSino-tunedSSIM-dropoutAdded-0p3lr-800epochs' 
#csv_file_name = 'frame-ACT-288-bilinear-288x257-padSino-tunedSSIM-dropoutEnforced-0p3lr-165epochs'
#csv_file_name = 'frame-ACT-288-bilinear-288x257-padSino-tunedSSIM-0p3lr-800epochs.csv'
csv_file_name = 'frame-ACT-320-bilinear-288x257-padSino-tunedSSIM-0p3lr-607epochs.csv'
#csv_file_name = 'frame-RECON_SINO_IS-320-FORE_recon-bilinear-288x257-padSino-tunedSSIM-0p3lr-50epochs.csv'

metric_name = 'SSIM'

metric_range=None
#metric_range=(0.905,0.915)
#metric_range=(0,400)

epoch_range=None
#epoch_range=(0,293)
#epoch_range=(0,65)
#epoch_range=(700,800)

#csv_dir_path = r"working/dataframes-train"
csv_dir_path = paths['train_dataframe_dirPath']
print(csv_dir_path)

plot_learning_curves(
    csv_dir_path,
    csv_file_name,
    metric_name,
    epoch_range=epoch_range,
    metric_range=metric_range, #(0.825,.9),
    train_label=None,
    holdout_label=None,
    qa_label=None,
    ax=None,
    title=None,
    fontsize=12,
    ticksize=10,
)

## Describe Test Dataframes

In [ ]:
test_dataframe_dirPath = paths['test_dataframe_dirPath']

bilinear_288x257_padZeros_1p0lr_trainSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x257-padZeros-tunedSSIM-1p0lr-600epochs-trainSet.csv'))
bilinear_288x257_padZeros_1p0lr_testSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x257-padZeros-tunedSSIM-1p0lr-600epochs-testSet.csv'))
pool_288x257_padZeros_1p0lr_trainSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-pool-288x257-padZeros-tunedSSIM-1p0lr-600epochs-trainSet.csv'))
pool_288x257_padZeros_1p0lr_testSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-pool-288x257-padZeros-tunedSSIM-1p0lr-600epochs-testSet.csv'))

bilinear_288x257_padZeros_0p3lr_trainSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x257-padZeros-tunedSSIM-0p3lr-800epochs-trainSet.csv'))
bilinear_288x257_padZeros_0p3lr_testSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x257-padZeros-tunedSSIM-0p3lr-800epochs-testSet.csv'))
pool_288x257_padZeros_0p3lr_trainSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-pool-288x257-padZeros-tunedSSIM-0p3lr-800epochs-trainSet.csv'))
pool_288x257_padZeros_0p3lr_testSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-pool-288x257-padZeros-tunedSSIM-0p3lr-800epochs-testSet.csv'))

bilinear_288x257_padSino_0p3lr_trainSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x257-padSino-tunedSSIM-0p3lr-800epochs-trainSet.csv'))
bilinear_288x257_padSino_0p3lr_testSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x257-padSino-tunedSSIM-0p3lr-800epochs-testSet.csv'))

bilinear_288x218_padSino_0p3lr_trainSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x218-padSino-tunedSSIM-0p3lr-800epochs-trainSet.csv'))
bilinear_288x218_padSino_0p3lr_testSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x218-padSino-tunedSSIM-0p3lr-800epochs-testSet.csv'))

bilinear_288x180_padSino_0p3lr_trainSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x180-padSino-tunedSSIM-0p3lr-800epochs-trainSet.csv'))
bilinear_288x180_padSino_0p3lr_testSet = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-288-bilinear-288x180-padSino-tunedSSIM-0p3lr-800epochs-testSet.csv'))

net320_train = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-320-bilinear-288x257-padSino-tunedSSIM-0p3lr-607epochs-trainSet.csv'))
net320_test = pd.read_csv(os.path.join(test_dataframe_dirPath, 'frame-ACT-320-bilinear-288x257-padSino-tunedSSIM-0p3lr-607epochs-testSet.csv'))

#pool_288x257_padZeros_1p0lr_trainSet.describe()
#pool_288x257_padZeros_1p0lr_testSet.describe()
# B
#bilinear_288x257_padZeros_1p0lr_trainSet.describe()
#bilinear_288x257_padZeros_1p0lr_testSet.describe()
# C
#pool_288x257_padZeros_0p3lr_trainSet.describe()
#pool_288x257_padZeros_0p3lr_testSet.describe()
# D
#bilinear_288x257_padZeros_0p3lr_trainSet.describe()
#bilinear_288x257_padZeros_0p3lr_testSet.describe()
# E
#bilinear_288x257_padSino_0p3lr_trainSet.describe()
#bilinear_288x257_padSino_0p3lr_testSet.describe()
# F
#bilinear_288x218_padSino_0p3lr_trainSet.describe()
#bilinear_288x218_padSino_0p3lr_testSet.describe()
# G
#bilinear_288x180_padSino_0p3lr_trainSet.describe()
#bilinear_288x180_padSino_0p3lr_testSet.describe()
# I
#net320_train.describe()
net320_test.describe()






## Describe Dataframes ##

#frame_picked = dataframe[dataframe["SSIM (ML-EM)"]>dataframe["SSIM (FBP)"]]


## Plot Test Dataframes

In [ ]:
#dataframe1 = bilinear_288x257-padZeros_1p0lr_trainSet
#dataframe1 = bilinear_288x257_padZeros_1p0lr_testSet
#dataframe2 = pool_288x257-padZeros_1p0lr_trainSet
#dataframe2 = pool_288x257_padZeros_1p0lr_testSet

#dataframe1=bilinear_288x257_padZeros_0p3lr_testSet
#dataframe2=bilinear_288x257_padSino_0p3lr_testSet

dataframe1=net320_train
dataframe2=net320_test

## Specify Plotting Parameters ##
plot_type = 2 # 1 = histograms, 2 = bin plots, 3 = both

#column_MSE_1 = 'MSE (Recon1)'
column_MSE_1 = 'MSE (Recon2)'
column_MSE_2 = 'MSE (Network)'

#column_SSIM_1 = 'SSIM (Recon1)'
column_SSIM_1 = 'SSIM (Recon2)'
column_SSIM_2 = 'SSIM (Network)'


MSE_lim_1D = (0,10)
MSE_lim_2D = (0,10)
SSIM_lim_1D = (0.8,1)
SSIM_lim_2D = (0.8,1)

ylim_1D = (0,600)
x_column_label=None
y_column_label=None

titlesize=12
fontsize=9
ticksize=8
dpi=800
fig_size_12 = (8,6)
fig_size_3 = (15,6)

if plot_type == 1 or plot_type == 2:
    figsize=fig_size_12 # 17,5
    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs = gridspec.GridSpec(ncols=100, nrows=100)

    # Top Row Axes #
    ax1 = fig.add_subplot(gs[0:42,   0:43])
    ax2 = fig.add_subplot(gs[0:42,   57:100])

    # Bottom Row Axes #
    ax3 = fig.add_subplot(gs[58:100, 0:43])
    ax4 = fig.add_subplot(gs[58:100, 57:100])

    if plot_type == 1:
        plot_hist_1D(ax1, dataframe1, '(1) CNN-A: MSE Histogram',  'MSE', 'frequency', column_MSE_1 , column_MSE_2, xlim=MSE_lim_1D, ylim=ylim_1D, bins=40)
        plot_hist_1D(ax2, dataframe1, '(2) CNN-A: SSIM Histogram', 'SSIM','frequency', column_SSIM_1, column_SSIM_2, xlim=SSIM_lim_1D, ylim=ylim_1D, bins=40)
        plot_hist_1D(ax3, dataframe2, '(3) CNN-B: MSE Histogram',  'MSE', 'frequency', column_MSE_1 , column_MSE_2,  xlim=MSE_lim_1D, ylim=ylim_1D,  bins=40)
        plot_hist_1D(ax4, dataframe2, '(4) CNN-B: SSIM Histogram', 'SSIM','frequency', column_SSIM_1, column_SSIM_2, xlim=SSIM_lim_1D, ylim=ylim_1D, bins=40)
    if plot_type == 2:
        plot_hist_2D(ax1, dataframe1, '(1) CNN-A: MSE Bin Plot', column_MSE_1, 'MSE (CNN-A)', column_MSE_1 , column_MSE_2,MSE_lim_2D, MSE_lim_2D,
                     gridsize=60, titlesize=titlesize, fontsize=fontsize, ticksize=ticksize, x_column_label=x_column_label, y_column_label=y_column_label)
        plot_hist_2D(ax2, dataframe1, '(2) CNN-A: SSIM Bin Plot',column_SSIM_1, 'SSIM (CNN-A)', column_SSIM_1, column_SSIM_2, SSIM_lim_2D, SSIM_lim_2D,
                     gridsize=100, titlesize=titlesize, fontsize=fontsize, ticksize=ticksize, x_column_label=x_column_label, y_column_label=y_column_label)
        plot_hist_2D(ax3, dataframe2, '(3) CNN-B: MSE Bin Plot', column_MSE_1, 'MSE (CNN-B)', column_MSE_1 , column_MSE_2, MSE_lim_2D, MSE_lim_2D,
                     gridsize=60, titlesize=titlesize, fontsize=fontsize, ticksize=ticksize, x_column_label=x_column_label, y_column_label=y_column_label)
        plot_hist_2D(ax4, dataframe2, '(4) CNN-B: SSIM Bin Plot', column_SSIM_1, 'SSIM (CNN-B)', column_SSIM_1, column_SSIM_2, SSIM_lim_2D, SSIM_lim_2D,
                     gridsize=100, titlesize=titlesize, fontsize=fontsize, ticksize=ticksize, x_column_label=x_column_label, y_column_label=y_column_label)

if plot_type == 3:
    figsize=figsize_3 # 17,5
    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs = gridspec.GridSpec(ncols=100, nrows=100)

    # Top Row Axes #
    ax1 = fig.add_subplot(gs[0:42,   0:18]) # 20
    ax2 = fig.add_subplot(gs[0:42,   25:47]) # 22
    ax3 = fig.add_subplot(gs[0:42,   53:74]) # 20
    ax4 = fig.add_subplot(gs[0:42,   80:100]) # 22

    # Bottom Row Axes #
    ax5 = fig.add_subplot(gs[58:100, 0:18]) # -5-
    ax6 = fig.add_subplot(gs[58:100, 25:47]) # -3 - -3-
    ax7 = fig.add_subplot(gs[58:100, 53:74]) # -5-
    ax8 = fig.add_subplot(gs[58:100, 80:100])

    plot_hist_1D(ax1, dataframe1, '(1) CNN-A: MSE Histogram',  'MSE', 'frequency', column_MSE_1 , column_MSE_2, xlim=MSE_lim_1D, ylim=ylim_1D, bins=40)
    plot_hist_1D(ax2, dataframe1, '(3) CNN-A: SSIM Histogram', 'SSIM','frequency', column_SSIM_1, column_SSIM_2, xlim=SSIM_lim_1D, ylim=ylim_1D, bins=40)
    plot_hist_2D(ax3, dataframe1, '(5) CNN-A: MSE Bin Plot', column_MSE_1, 'MSE (CNN-A)', column_MSE_1 , column_MSE_2,MSE_lim_2D, MSE_lim_2D, gridsize=60)
    plot_hist_2D(ax4, dataframe1, '(7) CNN-A: SSIM Bin Plot',column_SSIM_1, 'SSIM (CNN-A)', column_SSIM_1, column_SSIM_2, SSIM_lim_2D, SSIM_lim_2D, gridsize=100)

    plot_hist_1D(ax5, dataframe2, '(2) CNN-B: MSE Histogram',  'MSE', 'frequency', column_MSE_1 , column_MSE_2,  xlim=MSE_lim_1D, ylim=ylim_1D,  bins=40)
    plot_hist_1D(ax6, dataframe2, '(4) CNN-B: SSIM Histogram', 'SSIM','frequency', column_SSIM_1, column_SSIM_2, xlim=SSIM_lim_1D, ylim=ylim_1D, bins=40)
    plot_hist_2D(ax7, dataframe2, '(6) CNN-B: MSE Bin Plot', column_MSE_1, 'MSE (CNN-B)', column_MSE_1 , column_MSE_2, MSE_lim_2D, MSE_lim_2D, gridsize=60)
    plot_hist_2D(ax8, dataframe2, '(8) CNN-B: SSIM Bin Plot', column_SSIM_1, 'SSIM (CNN-B)', column_SSIM_1, column_SSIM_2, SSIM_lim_2D, SSIM_lim_2D, gridsize=100)

# Set Data

In [ ]:
display='sample'
outputs_to_plot=['act_image', 'cnn_output','recon1','recon2']

device = "cpu"
fig_scale=1.5

if display=='nema':
    act_sino_array_name = 'QA-NEMA-highCountSino-382x513.npy'
    act_image_array_name ='QA-NEMA-actMap.npy'
    atten_sino_array_name = 'QA-NEMA-attenSino-382x513.npy'
    atten_image_array_name = 'QA-NEMA-attenMap.npy'
    recon1_name = 'QA-NEMA-highCountImage.npy'
    recon2_name = 'QA-NEMA-obliqueImage.npy'
    hot_mask_13_name = 'QA-NEMA-hotMask_13mm.npy'
    hot_back_mask_13_name = 'QA-NEMA-backMask_13mm.npy'
    hot_mask_17_name = 'QA-NEMA-hotMask_17mm.npy'
    hot_back_mask_17_name = 'QA-NEMA-backMask_17mm.npy'
    indexes = [13, 14, 15, 16, 17, 18, 19]
    #indexes = list(range(35))
    hot_mask_13 = torch.from_numpy(np.load(os.path.join(paths['data_dirPath'], hot_mask_13_name), mmap_mode='r'))[indexes]
    hot_back_mask_13 = torch.from_numpy(np.load(os.path.join(paths['data_dirPath'], hot_back_mask_13_name), mmap_mode='r'))[indexes]
    
    hot_mask_17 = torch.from_numpy(np.load(os.path.join(paths['data_dirPath'], hot_mask_17_name), mmap_mode='r'))[indexes]
    hot_back_mask_17 = torch.from_numpy(np.load(os.path.join(paths['data_dirPath'], hot_back_mask_17_name), mmap_mode='r'))[indexes]
    fig_size = 1.8
if display=='sample':
    act_sino_array_name = 'samples-highCountSino-382x513.npy'
    act_image_array_name ='samples-actMap.npy'
    atten_sino_array_name = 'samples-attenSino-382x513.npy'
    atten_image_array_name = 'samples-attenMap.npy'
    recon1_name = 'samples-highCountImage.npy'
    recon2_name = 'samples-obliqueImage.npy'
    indexes = [0, 1, 2, 6, 7, 8, 9]
    fig_size = 1.8
if display=='test':
    act_sino_array_name = 'test-highCountSino-382x513.npy'
    act_image_array_name ='test-actMap.npy'
    atten_sino_array_name = 'test-attenSino-382x513.npy'
    atten_image_array_name = 'test-attenMap.npy'
    recon1_name = 'test-highCountImage.npy'
    recon2_name = 'test-obliqueImage.npy'
    indexes = [1200, 1300, 1400, 2350, 3800, 4360]
    fig_size = 1.8
if display=='train':
    act_sino_array_name = 'train-highCountSino-382x513.npy'
    act_image_array_name ='train-actMap.npy'
    atten_sino_array_name = 'train-attenSino-382x513.npy'
    atten_image_array_name = 'train-attenMap.npy'    
    recon1_name = 'train-highCountImage.npy'
    recon2_name = 'train-obliqueImage.npy'
    indexes = [1100, 1200, 1300, 1600, 1700, 1800, 1900]
    #indexes = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
    fig_size = 1.8
if display=='radial':
    act_sino_array_name = 'QA-Radial-highCountSino-382x513.npy'
    act_image_array_name ='QA-Radial-actMap.npy'
    atten_sino_array_name = 'QA-Radial-attenSino-382x513.npy'
    atten_image_array_name = 'QA-Radial-attenMap.npy'
    recon1_name = 'QA-Radial-highCountImage.npy'
    recon2_name = 'QA-Radial-obliqueImage.npy'
    indexes = [5, 15, 25]
    fig_size = 2
if display=='pinwheel':
    act_sino_array_name = 'QA-Pinwheel-highCountSino-382x513.npy'
    act_image_array_name ='QA-Pinwheel-actMap.npy'
    atten_sino_array_name = 'QA-Pinwheel-attenSino-382x513.npy'
    atten_image_array_name = 'QA-Pinwheel-attenMap.npy'
    recon1_name = 'QA-Pinwheel-highCountImage.npy'
    recon2_name = 'QA-Pinwheel-obliqueImage.npy'
    indexes = [5, 15, 25]
    fig_size = 2.5
if display=='axial':
    act_sino_array_name = 'QA-Axial-highCountSino-382x513.npy'
    act_image_array_name ='QA-Axial-actMap.npy'
    atten_sino_array_name = 'QA-Axial-attenSino-382x513.npy'
    atten_image_array_name = 'QA-Axial-attenMap.npy'
    recon1_name = 'QA-Axial-highCountImage.npy'
    recon2_name = 'QA-Axial-obliqueImage.npy'
    indexes = [5, 10, 15, 20, 25, 30]
    fig_size = 1.8

if display=='mixed1':
    act_sino_array_name = 'train-highCountSino-382x513.npy'
    act_image_array_name ='train-actMap.npy' 

    atten_sino_array_name = 'QA-NEMA-attenSino-382x513.npy'
    atten_image_array_name = 'QA-NEMA-attenMap.npy'
    
    recon1_name = 'train-highCountImage.npy'
    recon2_name = 'train-obliqueImage.npy'
    indexes = [5, 10, 15, 20, 25, 30]
    fig_size = 1.8
if display=='mixed2':
    act_sino_array_name = 'train-highCountSino-382x513.npy'
    act_image_array_name ='train-actMap.npy'

    atten_sino_array_name = 'QA-Radial-attenSino-382x513.npy'
    atten_image_array_name = 'QA-Radial-attenMap.npy'
    
    recon1_name = 'train-highCountImage.npy'
    recon2_name = 'train-obliqueImage.npy'
    indexes = [5, 10, 15, 20, 25, 30]
    fig_size = 1.8
    
if display=='mixed3':
    act_sino_array_name = 'QA-Radial-highCountSino-382x513.npy'
    act_image_array_name ='QA-Radial-actMap.npy'
    atten_sino_array_name = 'QA-Radial-attenSino-382x513.npy'

    atten_sino_array_name = 'QA-Pinwheel-attenSino-382x513.npy'
    atten_image_array_name = 'QA-Pinwheel-attenMap.npy'
    
    recon1_name = 'QA-Radial-highCountImage.npy'
    recon2_name = 'QA-Radial-obliqueImage.npy'
    indexes = [5, 15, 25]
    fig_size = 1.4


# Plot Images: Newer Networks (288x288 only)

## (A) ACT, pool-288x257, padZeros, no LR decay (600 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-288-pool-288x257-padZeros-tunedSSIM-1p0lr-600epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = { 
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 10,
  "SI_gen_mult": 1.5012782419950113,
  "SI_gen_neck": "narrow", # narrow
  "SI_gen_z_dim": 872,
  "SI_half_life_examples": -1, 
  "SI_layer_norm": "instance", # instance
  "SI_learnedScale_init": 7.305980552864529,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.6943444827125673,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "conv",
  "SI_stats_criterion": -1,
  "SI_moment_1_fraction": -1,
  "batch_base2_exponent": 7,
  "gen_b1": 0.3600790033157822,
  "gen_b2": 0.6033159868492163,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0024018267054557695,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 288, # 288
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='pool',
                  sino_pad_type='zeros',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=2, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=257) # Not used

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)


## (B) ACT, bilinear-288x257, padZeros, no LR decay (600 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-288-bilinear-288x257-padZeros-tunedSSIM-1p0lr-600epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI= {
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "Tanh",
  "SI_gen_hidden_dim": 10,
  "SI_gen_mult": 1.9886575078445312,
  "SI_gen_neck": "narrow",
  "SI_gen_z_dim": 1494,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "none",
  "SI_learnedScale_init": 5.894328507562502,
  "SI_moment_1_fraction": -1,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 9.862580591354153,
  "SI_pad_mode": "replicate",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "gen_b1": 0.5583313727913738,
  "gen_b2": 0.3819733954553099,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0016547324637469305,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 288,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='zeros',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=1, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=(288,257))

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)


## (C) ACT, pool-288x257, padZeros, 0.3 LR decay (800 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-288-pool-288x257-padZeros-tunedSSIM-0p3lr-800epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = { 
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 10,
  "SI_gen_mult": 1.5012782419950113,
  "SI_gen_neck": "narrow", # narrow
  "SI_gen_z_dim": 872,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance", # instance
  "SI_learnedScale_init": 7.305980552864529,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.6943444827125673,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "conv",
  "SI_stats_criterion": -1,
  "SI_moment_1_fraction": -1,
  "batch_base2_exponent": 7,
  "gen_b1": 0.3600790033157822,
  "gen_b2": 0.6033159868492163,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0024018267054557695,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 288, # 288
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='pool',
                  sino_pad_type='zeros',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=2, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=257) # Not used

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)


## (D) ACT, bilinear-288x257, padZeros, 0.3 LR decay (800 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-288-bilinear-288x257-padZeros-tunedSSIM-0p3lr-800epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI= {
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "Tanh",
  "SI_gen_hidden_dim": 10,
  "SI_gen_mult": 1.9886575078445312,
  "SI_gen_neck": "narrow",
  "SI_gen_z_dim": 1494,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "none",
  "SI_learnedScale_init": 5.894328507562502,
  "SI_moment_1_fraction": -1,
  "SI_normalize": False, 
  "SI_output_scale_lr_mult": 9.862580591354153,
  "SI_pad_mode": "replicate",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "gen_b1": 0.5583313727913738,
  "gen_b2": 0.3819733954553099,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0016547324637469305,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 288,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='zeros',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=1, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=(288,257))

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)


## (E) ACT, bilinear-288x257, padSino, 0.3 LR decay (800 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-288-bilinear-288x257-padSino-tunedSSIM-0p3lr-800epochs'
#checkpoint_name = 'checkpoint-ACT-288-bilinear-288x257-padSino-tunedSSIM-0p3lr-370epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = { 
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 20,
  "SI_gen_mult": 2.1572554323300173,
  "SI_gen_neck": "wide",
  "SI_gen_z_dim": 1939,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 4.981741020141211,
  "SI_moment_1_fraction": -1,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 4.290911545096751,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "conv",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "gen_b1": 0.19220905219147658,
  "gen_b2": 0.1767702014809058,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.000330514188658911,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 288,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}


tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sino',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=1, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=(288,257))

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)

## (F) ACT, bilinear-288x218, padSino, 0.3 LR decay (800 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-288-bilinear-288x218-padSino-tunedSSIM-0p3lr-800epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = { 
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 4,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 34,
  "SI_gen_mult": 1.6784499724995354,
  "SI_gen_neck": "wide",
  "SI_gen_z_dim": 1733,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 3.1060983465329053,
  "SI_moment_1_fraction": -1,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 4.134778822027846,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "gen_b1": 0.6099253222709485,
  "gen_b2": 0.2194047349693382,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0003521818874320586,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 288,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}


tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sino',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=1, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=(288,180))

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)

## (G) ACT, bilinear-288x180, padSino, 0.3 LR decay (800 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-288-bilinear-288x180-padSino-tunedSSIM-0p3lr-800epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = { 
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": None,
  "SI_gen_hidden_dim": 16,
  "SI_gen_mult": 2.3555295003025276,
  "SI_gen_neck": "medium",
  "SI_gen_z_dim": 576,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 10.893989027618026,
  "SI_moment_1_fraction": -1,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.1618819638475886,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "conv",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "gen_b1": 0.8174588820756479,
  "gen_b2": 0.7294818348374883,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0001232534342696305,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 288,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sino',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=1, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=(288,180))

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)

## (H) ACT, bilinear-288x257, padSino, 0.3 LR decay, dropout (165 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-288-bilinear-288x257-padSino-tunedSSIM-dropoutEnforced-0p3lr-165epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = { 
  "SI_alpha_min": -1,
  "SI_dropout": True,
  "SI_exp_kernel": 4,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": None,
  "SI_gen_hidden_dim": 17,
  "SI_gen_mult": 3.235794135077702,
  "SI_gen_neck": "medium",
  "SI_gen_z_dim": 776,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 0.6598989774160382,
  "SI_moment_1_fraction": -1,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.1381973993890924,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "conv",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "frozen_variant": "RECON_SINO",
  "gen_b1": 0.6315686454122481,
  "gen_b2": 0.4796586315253887,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.004073671882171635,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 288,
  "network_type": "ACT",
  "recon_variant": 1,
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}


tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sino',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=1, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=(288,257))

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(act_sino)
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)

## (I) ACT 320, bilinear-288x257, padSino, 0.3 LR decay, (800 epochs)

In [ ]:
checkpoint_name = 'checkpoint-ACT-320-bilinear-288x257-padSino-tunedSSIM-0p3lr-607epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = {
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 27,
  "SI_gen_mult": 2.346554155198677,
  "SI_gen_neck": "narrow",
  "SI_gen_z_dim": 1275,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "none",
  "SI_learnedScale_init": 22.311191399467425,
  "SI_moment_1_fraction": -1,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.1652951981214295,
  "SI_pad_mode": "replicate",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "frozen_variant": "RECON_SINO",
  "gen_b1": 0.16376752129041514,
  "gen_b2": 0.6314511499737464,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.00021715942590932197,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 320,
  "network_type": "ACT",
  "recon_variant": 1,
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}


tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sino',
                  image_pad_type='zeros',
                  sino_init_vert_cut=288,  # 320. However, this is irrelevant for sino_resize_type='pool'
                  vert_pool_size=1,  # Final vertical size: 382--initial cut-->320-->final cut 288
                  horiz_pool_size=1, # Final horizontal size: (513+1)/2 = 257
                  bilinear_intermediate_size=(288,257))

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + (ACT IMAGE/Mask)')
print('=============================')
show_single_unmatched_tensor(act_sino)
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
MSE_Oblique = calculate_metric(recon2, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_FORE = calculate_metric(recon1, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)
MSE_CNN = calculate_metric(cnn_output, act_image, MSE, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)
print('MSE (Oblique):', MSE_Oblique)
print('MSE (FORE):', MSE_FORE)
print('MSE (CNN):', MSE_CNN)


if display=='nema':
    CRC_13_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_13, hot_mask_13)    
    CRC_13_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_13, hot_mask_13)
    CRC_13_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_13, hot_mask_13)
    
    CRC_17_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_back_mask_17, hot_mask_17)    
    CRC_17_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_back_mask_17, hot_mask_17)
    CRC_17_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_back_mask_17, hot_mask_17)

    print('CRC 13 mm (Oblique): ', CRC_13_score_Oblique)
    print('CRC 17 mm (Oblique): ', CRC_17_score_Oblique)
    print('CRC 13 mm (FORE): ', CRC_13_score_FORE)
    print('CRC 17 mm (FORE): ', CRC_17_score_FORE)
    print('CRC 13 mm (CNN): ', CRC_13_score_CNN)
    print('CRC 17 mm (CNN): ', CRC_17_score_CNN)

# Plot Images: Older Networks

## 256x256, Tuned s/ Stats

In [ ]:
checkpoint_name = 'checkpoint-ACT-256-widePadSino-fill_1_skipNone-tunedStats-300epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']
network_type='ACT'

# Tuned for stats metric (moment weights = [0.9, 0.1]
config_ACT_SI = {
  "SI_alpha_min": 0.5213698467044514,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 1,
  "SI_gen_final_activ": "ELU",
  "SI_gen_hidden_dim": 16,
  "SI_gen_mult": 1.6773364207779211,
  "SI_gen_neck": "wide",
  "SI_gen_z_dim": 727,
  "SI_half_life_examples": 3151.6045889459424,
  "SI_layer_norm": "group",
  "SI_learnedScale_init": 12.392331521009083,
  "SI_moment_1_fraction": 0.46359052811993384,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.2702660653043705,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "none",
  "SI_stats_criterion": "PatchwiseMomentLoss",
  "batch_base2_exponent": 5,
  "gen_b1": 0.2713979350750665,
  "gen_b2": 0.8655007369238326,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.00043162727752556266,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 256,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, network_type,
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = atten_image_array_name,
                  atten_sino_array_name = atten_sino_array_name,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sinogram',
                  image_pad_type='zeros',
                  sino_init_vert_cut=300,
                  vert_pool_size=1,
                  horiz_pool_size=1,
                  bilinear_intermediate_size=180)

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + ACT IMAGE')
print('======================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)


SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)

if display=='nema':
    CRC_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_mask_17, hot_back_mask_17)    
    CRC_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_mask_17, hot_back_mask_17)
    CRC_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_mask_17, hot_back_mask_17)
    print('CRC (Oblique): ', CRC_score_Oblique)
    print('CRC (FORE): ', CRC_score_FORE)
    print('CRC (CNN): ', CRC_score_CNN)

## 256x256, Grokking?

In [ ]:
checkpoint_name = 'checkpoint-ACT-256-widePadSino-fill_1_skipNone-tunedSSIM-300epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

network_type='ACT'


config_ACT_SI ={ # 256x256, bilinear intemediate size = 180, tuned for SSIM, pad_type='sinogram', fill enforced to =1
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 1,
  "SI_gen_final_activ": None,
  "SI_gen_hidden_dim": 15,
  "SI_gen_mult": 2.8577974008839018,
  "SI_gen_neck": "medium",
  "SI_gen_z_dim": 1824,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 4.0327009143861074,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 2.9521606233259443,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 7,
  "gen_b1": 0.6207299459912765,
  "gen_b2": 0.5773966392271939,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.00026018975910785543,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 256,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, network_type,
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = atten_image_array_name,
                  atten_sino_array_name = atten_sino_array_name,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sinogram',
                  image_pad_type='zeros',
                  sino_init_vert_cut=300,
                  vert_pool_size=1,
                  horiz_pool_size=1,
                  bilinear_intermediate_size=180)

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + ACT IMAGE')
print('======================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)


SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)

if display=='nema':
    CRC_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_mask_17, hot_back_mask_17)    
    CRC_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_mask_17, hot_back_mask_17)
    CRC_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_mask_17, hot_back_mask_17)
    print('CRC (Oblique): ', CRC_score_Oblique)
    print('CRC (FORE): ', CRC_score_FORE)
    print('CRC (CNN): ', CRC_score_CNN)


## 256x256, FROZEN_COUNTERFLOW

In [ ]:
checkpoint_name = 'checkpoint-FROZEN_COUNTERFLOW-256-untuned-100epochs'
network_type='FROZEN_COUNTERFLOW'
outputs_to_plot=['act_image','recon1','cnn_output', 'atten_image']

#atten_image_array_name = 'QA-Radial-attenMap.npy'
#atten_image_array_name = 'QA-Radial-attenMap-2xWater.npy'
#atten_image_array_name = 'QA-Radial-attenMap-swappedLACs.npy'


config_FROZEN_COUNTERFLOW= { # SAMPLE
  "FROZEN_IS_alpha_min": -1,
  "FROZEN_IS_dropout": False,
  "FROZEN_IS_exp_kernel": 3,
  "FROZEN_IS_fixedScale": 1,
  "FROZEN_IS_gen_fill": 0,
  "FROZEN_IS_gen_final_activ": None,
  "FROZEN_IS_gen_hidden_dim": 15,
  "FROZEN_IS_gen_mult": 3.3144387875060906,
  "FROZEN_IS_gen_neck": "medium",
  "FROZEN_IS_gen_z_dim": 1835,
  "FROZEN_IS_half_life_examples": -1,
  "FROZEN_IS_layer_norm": "group",
  "FROZEN_IS_learnedScale_init": 18.411171440894215,
  "FROZEN_IS_normalize": False,
  "FROZEN_IS_output_scale_lr_mult": 1.7607516297239543,
  "FROZEN_IS_pad_mode": "zeros",
  "FROZEN_IS_skip_mode": "none",
  "FROZEN_IS_stats_criterion": -1,
  "FROZEN_batch_base2_exponent": 6,
  "FROZEN_gen_b1": 0.17828464968859092,
  "FROZEN_gen_b2":  0.22254220083596676,
  "FROZEN_gen_image_channels": 1,
  "FROZEN_gen_image_size": 180,
  "FROZEN_gen_lr": 0.00011584402663085701,
  "FROZEN_gen_sino_channels_SI": 1,
  "FROZEN_gen_sino_size": 256,
  "FROZEN_network_type": "ATTEN",
  "FROZEN_sup_base_criterion": "MSELoss",
  "FROZEN_train_SI": False,
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": None,
  "SI_gen_hidden_dim": 15,
  "SI_gen_mult": 3.3144387875060906,
  "SI_gen_neck": "medium",
  "SI_gen_z_dim": 1835,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "group",
  "SI_learnedScale_init": 18.411171440894215,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.7607516297239543,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 6, #6
  "gen_b1": 0.17828464968859092,
  "gen_b2": 0.22254220083596676,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.00011584402663085701,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 256,
  "network_type": "FROZEN_COUNTERFLOW",
  "frozen_variant": "ATTEN",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, network_type,
                  config_FROZEN_COUNTERFLOW, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = atten_image_array_name,
                  atten_sino_array_name = atten_sino_array_name,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sinogram',
                  image_pad_type='zeros',
                  sino_init_vert_cut=300,
                  vert_pool_size=1,
                  horiz_pool_size=1,
                  bilinear_intermediate_size=180)

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + ACT IMAGE')
print('======================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)

if display=='nema':
    CRC_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_mask_17, hot_back_mask_17)    
    CRC_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_mask_17, hot_back_mask_17)
    CRC_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_mask_17, hot_back_mask_17)
    print('CRC (Oblique): ', CRC_score_Oblique)
    print('CRC (FORE): ', CRC_score_FORE)
    print('CRC (CNN): ', CRC_score_CNN)

## 256x256, CONCAT

In [ ]:
checkpoint_name = 'checkpoint-CONCAT-256-widePadSino-tunedSSIM-90epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output', 'atten_image']
network_type='CONCAT'

config_CONCAT = {
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 4,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 29,
  "SI_gen_mult": 2.031696618359278,
  "SI_gen_neck": "medium",
  "SI_gen_z_dim": 958,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 6.099856735304703,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 2.166354001308141,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "gen_b1": 0.3706466944358945,
  "gen_b2": 0.4364991709587488,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0004300814461678685,
  "gen_sino_channels_SI": 4,
  "gen_sino_size": 256,
  "network_type": "CONCAT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}


tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, network_type,
                  config_CONCAT, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = atten_sino_array_name,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sinogram',
                  image_pad_type='zeros',
                  sino_init_vert_cut=300,
                  vert_pool_size=1,
                  horiz_pool_size=1,
                  bilinear_intermediate_size=180)

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']

show_single_unmatched_tensor(act_sino[0:2], fig_size=30)

SSIM_score1 = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_score2 = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_score3 = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)


print("=====================")
print('SSIM (CNN):', SSIM_score1)
print('SSIM (FORE):', SSIM_score2)
print('SSIM (Oblique):', SSIM_score3)

if display=='NEMA':
    CRC_score =  ROI_NEMA_hot(act_image, cnn_output, hot_mask_17, hot_back_mask_17)
    print('CRS (CNN):', CRC_score)

## 256x256, wide sinogram padding

In [ ]:
#checkpoint_name = 'checkpoint-ACT-256-largePadSino-tunedSSIM-100epochs'
checkpoint_name = 'checkpoint-ACT-256-widePadSino-tunedSSIM-100epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = { # 256x256, tuned SSIM, pad_type='sinogram', bilinear intermediate size = 180
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": None,
  "SI_gen_hidden_dim": 15,
  "SI_gen_mult": 3.3144387875060906,
  "SI_gen_neck": "medium",
  "SI_gen_z_dim": 1835,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "group",
  "SI_learnedScale_init": 18.411171440894215,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.7607516297239543,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 6,
  "gen_b1": 0.17828464968859092,
  "gen_b2": 0.22254220083596676,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.00011584402663085701,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 256,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = atten_image_array_name,
                  atten_sino_array_name = atten_sino_array_name,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='sinogram',
                  image_pad_type='zeros',
                  sino_init_vert_cut=300,
                  vert_pool_size=1,
                  horiz_pool_size=1,
                  bilinear_intermediate_size=180)

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + ACT IMAGE')
print('======================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)

if display=='nema':
    CRC_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_mask_17, hot_back_mask_17)    
    CRC_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_mask_17, hot_back_mask_17)
    CRC_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_mask_17, hot_back_mask_17)
    print('CRC (Oblique): ', CRC_score_Oblique)
    print('CRC (FORE): ', CRC_score_FORE)
    print('CRC (CNN): ', CRC_score_CNN)

## 180x180, bilinear, narrow zeros padding

In [ ]:
checkpoint_name = 'checkpoint-ACT-180-narrowPadZeros-tunedSSIM-100epochs'
outputs_to_plot=['act_image','recon2','recon1','cnn_output']

config_ACT_SI = { # 180x180, tuned SSIM, pad_type='zeros', bilinear_intermediate_size = 161, horiz_pool=2, vert_pool=1
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 4,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 11,
  "SI_gen_mult": 2.0282722914428213,
  "SI_gen_neck": "narrow",
  "SI_gen_z_dim": 584,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 10.553559972734485,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 8.240938610220685,
  "SI_pad_mode": "replicate",
  "SI_skip_mode": "conv",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 5,
  "gen_b1": 0.4495215605123463,
  "gen_b2": 0.15053718115803394,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0003521542451328137,
  "gen_sino_channels_SI": 3,
  "gen_sino_size": 180,
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, 'ACT',
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = None,
                  atten_sino_array_name = None,
                  recon1_array_name = recon1_name,
                  recon2_array_name = recon2_name,
                  sino_resize_type='bilinear',
                  sino_pad_type='zeros',
                  image_pad_type='zeros',
                  sino_init_vert_cut=320,
                  vert_pool_size=1,
                  horiz_pool_size=2,
                  bilinear_intermediate_size=161)


act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + ACT IMAGE')
print('======================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)

SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)

if display=='nema':
    CRC_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_mask_17, hot_back_mask_17)    
    CRC_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_mask_17, hot_back_mask_17)
    CRC_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_mask_17, hot_back_mask_17)
    print('CRC (Oblique): ', CRC_score_Oblique)
    print('CRC (FORE): ', CRC_score_FORE)
    print('CRC (CNN): ', CRC_score_CNN)

## 256x256, ATTEN, sino->image

In [ ]:
checkpoint_name = 'checkpoint-ATTEN_SI-256-widePadSino-untuned-25epochs'
network_type='ATTEN'

config_ATTEN_SI= { # SAMPLE
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": None,
  "SI_gen_hidden_dim": 15,
  "SI_gen_mult": 3.3144387875060906,
  "SI_gen_neck": "medium",
  "SI_gen_z_dim": 1835,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "group",
  "SI_learnedScale_init": 18.411171440894215,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.7607516297239543,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "none",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 6,
  "gen_b1": 0.17828464968859092,
  "gen_b2": 0.22254220083596676,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.00011584402663085701,
  "gen_sino_channels_SI": 1,
  "gen_sino_size": 256,
  "network_type": "ATTEN",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

tensors, cnn_output = PlotPhantomRecons(indexes, checkpoint_name, network_type,
                  config_ATTEN_SI, paths, fig_size, device, settings,
                  outputs_to_plot = outputs_to_plot,
                  act_image_array_name = atten_image_array_name,
                  act_sino_array_name = act_sino_array_name,
                  atten_image_array_name = atten_image_array_name,
                  atten_sino_array_name = atten_sino_array_name,
                  recon1_array_name = None,
                  recon2_array_name = None,
                  sino_resize_type='bilinear',
                  sino_pad_type='sinogram',
                  image_pad_type='zeros',
                  sino_init_vert_cut=300,
                  vert_pool_size=1,
                  horiz_pool_size=1,
                  bilinear_intermediate_size=180)

act_sino = tensors['act_sino_tensor']
act_image = tensors['act_image_tensor']
recon1 = tensors['recon1_tensor']
recon2 = tensors['recon2_tensor']
atten_image = tensors['atten_image_tensor']
#mask = np.load(r'C:/datast-sets/


print('CNN OUTPUT + ACT IMAGE')
print('======================')
show_single_unmatched_tensor(cnn_output, fig_size=fig_size*2)
show_single_unmatched_tensor(act_image, fig_size=fig_size*2)
show_single_unmatched_tensor(5*cnn_output + 1*act_image, fig_size=fig_size*2)



SSIM_CNN = calculate_metric(cnn_output, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_FORE = calculate_metric(recon1, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)
SSIM_Oblique = calculate_metric(recon2, act_image, SSIM, return_dataframe=False, label='default', crop_factor=1)

print("=====================")
print('SSIM (Oblique):', SSIM_Oblique)
print('SSIM (FORE):', SSIM_FORE)
print('SSIM (CNN):', SSIM_CNN)

if display=='nema':
    CRC_score_Oblique =  ROI_NEMA_hot(act_image, recon2, hot_mask_17, hot_back_mask_17)    
    CRC_score_FORE =  ROI_NEMA_hot(act_image, recon1, hot_mask_17, hot_back_mask_17)
    CRC_score_CNN =  ROI_NEMA_hot(act_image, cnn_output, hot_mask_17, hot_back_mask_17)
    print('CRC (Oblique): ', CRC_score_Oblique)
    print('CRC (FORE): ', CRC_score_FORE)
    print('CRC (CNN): ', CRC_score_CNN)

## Compare Recons

# Network Notes

## 256x256, Tune by stats metric w/ stats loss: deep network, trained 300 epochs

training set (SSIM): 0.9106342097123464

test set (SSIM: 0.8878521058294507

NEMA (SSIM): 0.8229745221989495


## 256X256, Grokking: deep network, trained 300 epochs

training set (SSIM): 0.9260909110307695

test set (SSIM): 0.8962109171681936

NEMA (SSIM): 0.8079071172646114

NEMA (CRC): 

## 256x256, Frozen Counterflow

training set (SSIM): 0.9320924265517129 

test set (SSIM): 0.9104165567292107

NEMA (SSIM): 0.7954177600996835

## 256X256, Concatenated (ACT+ATTEN)

training set (SSIM): 0.9194957282808092

test set  (SSIM): 0.9063458773824903

NEMA (SSIM): 0.823534403528486 (CNN): 

## 256X256 Network, wide sinogram padding

Note: due to the good performance of this network, all susequent networks (above this), use the same input data: vertical crop to 300, bilinear resizing to 180. This is the data files (180x180). After data is loaded, it is augmented and then sinogram padded horizontally and zero padded vertically to 256x256.

### 100 epochs:

training set (SSIM)': 0.9145165756344795

test set (SSIM): 0.8924144506454467

NEMA (SSIM): 0.8422211110591888

NEMA (CRS): 5.4327197

### 10 epochs:

training set (SSIM): 0.8859575778245926

test set (SSIM): 0.8461712673306464

NEMA (SSIM): 0.8121777496167591

NEMA (CRS): 7.2759275

## 288x288, pooled, narrow zeros padding
Sinograms were created by vertical cropping to size 288, then pooling to size 257. Finally it was zero padded to width of 288

SSIM for some sample phantoms:

training set (SSIM): 0.8996513456106185

test set (SSIM): 0.8711688712239266

NEMA (SSIM): 0.7882774812834603

NEMA (CRS): -8.62283

## 180x180, bilinear, narrow zeros padding
Sinograms were created by vertical cropping to size 320, then resizing bilinearly to 161x161. Finally, it was zero padded to 180x180. Padding fraction was the same as 288x288 network.

training set (SSIM): 0.8836602836847306

test set (SSIM): 0.87947369068861

NEMA (SSIM): 0.8155272219862257

NEMA (CRS): 32.660236

## 256x256, Attenuation, sino->image

training (SSIM): 0.9756443401177725

test set (SSIM): 0.9684655484226016

NEMA (SSIM): 0.979976251721382

# Reload Package After Code Changes

If you modify package code, use this cell to reload without restarting the kernel:

In [ ]:
# Reload package after making code changes
reload_package()  # Reloads from local repo

# Or use refresh_repo() to pull from git and reinstall
#refresh_repo(IN_COLAB=IN_COLAB, force_fresh_clone=True)

# Rebuild dictionaries with updated code
all_dicts = build_all_dicts(params)
config = all_dicts['config']
paths = all_dicts['paths']
settings = all_dicts['settings']

print("✅ Package reloaded and dictionaries rebuilt!")

# Run Pipeline

In [ ]:
# Run the pipeline (function available from environment setup)
run_pipeline=False
if run_pipeline=True:

    run_pipeline(
        config=config,
        paths=paths,
        settings=settings,
        tune_opts=tune_opts,
        base_dirs=base_dirs,
        test_opts=test_opts,
    )